In [ ]:
#working env
import os
os.chdir(r'W:\snRNAseq_Python_env_normalMT')
os.listdir()

In [ ]:
#!pip3 install sc-toolbox
#!pip install anndata2ri
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
import random
import time
import pandas as pd
#import sc_toolbox
if os.name == 'nt': # only for Windows
    os.environ["R_HOME"] = r"C:\Programme\R\R-4.3.2"
    
import rpy2.rinterface_lib.callbacks
import anndata2ri
import logging

from rpy2.robjects import pandas2ri
from rpy2.robjects import r

sc.settings.verbosity = 0
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

import warnings
warnings.simplefilter('ignore')

In [ ]:
%%R
options(warn=-1)
library(MAST)

In [ ]:
adata_raw = sc.read("integrated_harmony_Ms_subset.h5ad")
adata_raw.X=adata_raw.layers['soupX_counts'].copy()
adata_raw

In [ ]:
df = adata_raw.var['gene_ids']

In [ ]:
type(df)

In [ ]:
 df.to_excel('background_genes.xlsx')

In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
np.max(adata_raw.X)

In [ ]:
np.min(adata_raw.X)

In [ ]:
type(adata_raw.layers['soupX_counts'])

In [ ]:
adata_raw.layers["logcounts"] = adata_raw.X
adata_raw

In [ ]:
# Assuming your DataFrame is named adata_raw and the column is named batch_key
adata_raw.obs['batch'] = adata_raw.obs['batch'].replace({'0': 'HFD_L1', '1': 'HFD'})


In [ ]:
adata_raw.obs[batch_key].value_counts()

In [ ]:
keep_batches = ["HFD", "HFD_L1"]
adata = adata_raw[adata_raw.obs[batch_key].isin(keep_batches)].copy()
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata

In [ ]:
adata.var["feature_types"].value_counts()

In [ ]:
adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()
adata

In [ ]:
np.max(adata.X)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

In [ ]:
np.max(adata.X)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts")
p2 = sns.histplot(adata.X.sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("Shifted logarithm")
plt.show()

In [ ]:
print(adata.obs['leiden_res0_2'])

In [ ]:
 sc.pl.umap(adata, color=["cell_type", "batch", 'Acta2', 'Spp1', 'Fth1', 'Mgp', 'Gpnmb', 'Ccnd3', 'Rnf213', 'Taco1', 'Mndal', 'Fkbp5', 'Trim30a', "Stat1", "Stat2", "Nlrc5", "Ifnar1" ])

In [ ]:
adata.obs["cell_type"] = [ct.replace("+", "") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("/", "-") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace(" ", "-") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("_", "-") for ct in adata.obs["cell_type"]]


In [ ]:
adata.obs["batch"] = adata.obs["batch"].astype("category")
adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")

In [ ]:
adata

In [ ]:
def prep_anndata(adata_):
    def fix_dtypes(adata_):
        df = pd.DataFrame(adata_.X.A, index=adata_.obs_names, columns=adata_.var_names)
        df = df.join(adata_.obs)
        return sc.AnnData(df[adata_.var_names], obs=df.drop(columns=adata_.var_names))

    adata_ = fix_dtypes(adata_)
    sc.pp.filter_genes(adata_, min_cells=3)
    return adata_

In [ ]:
from matplotlib.pyplot import rc_context
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
adata
adata=adata[adata.obs.cell_type != "NK-T-Cells"]

In [ ]:
adata.obs.rename({"batch": "Treatment"}, axis=1, inplace=True)
adata.obs.rename({"cell_type": "Cell Types"}, axis=1, inplace=True)
adata.obs["Treatment"].replace({"HFD": "HFD+vehicle", "HFD_L1": "HFD+msR4M-L1"}, inplace=True)
adata

In [ ]:
sc.set_figure_params(color_map="viridis", fontsize=12)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12


In [ ]:
palette = sns.color_palette("Set1")

# Extract blue and orange from the palette
blue = palette[1]
orange = palette[4]
color_key=[orange, blue]

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="Cell Types", method="wilcoxon"
)

In [ ]:
sc.set_figure_params(color_map="viridis", fontsize=18)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 16
with rc_context({'figure.figsize': (6,3)}):
    sc.pl.rank_genes_groups_dotplot(
        adata, groupby="Cell Types", standard_scale="var", n_genes=10, color_map="viridis",
        save="_Fig3_suppl_rank_genes_groups_dotplot.pdf"
)

In [ ]:
cell_type_marker=sc.get.rank_genes_groups_df(
    adata, group=None
)

In [ ]:
# Calculate percent expressed per gene per cluster
def percent_expressed(adata, groupby, genes):
    result = []
    for group in adata.obs[groupby].cat.categories:
        idx = adata.obs[groupby] == group
        data = adata[idx, genes].X
        # If sparse, convert to dense
        if hasattr(data, "toarray"):
            data = data.toarray()
        pct = (data > 0).sum(axis=0) / data.shape[0] * 100
        result.append(pd.DataFrame({
            "group": group,
            "names": genes,
            "pct_expr": pct
        }))
    return pd.concat(result, ignore_index=True)

# Get unique gene names from marker table
genes = cell_type_marker["names"].unique()
pct_expr_df = percent_expressed(adata, "Cell Types", genes)

# Merge with marker table
cell_type_marker = cell_type_marker.merge(
    pct_expr_df, how="left", on=["group", "names"]
)

In [ ]:
import os

# Create the folder if it doesn't exist
os.makedirs("Supplementary_Tables", exist_ok=True)

# Save the DataFrame as an Excel file in the new folder
cell_type_marker.to_excel("Supplementary_Tables/cell_type_marker_Ms.xlsx", index=False)


In [ ]:
with rc_context({"figure.figsize": (4, 3)}):
 plt.rcParams['font.family'] = 'Arial'
 plt.rcParams['font.size'] = 12
 sc.pl.umap(adata, color=["Cell Types","Treatment","Trem2", 'Stat1', "Stat2", "Ifnar1"], ncols=2, frameon=False, wspace=0.1, save="Fig3_umap_markers_Ms.pdf")

In [ ]:
sc.pl.umap(adata, color=["Cell Types","Treatment","Trem2", 'Stat1', "Stat2", "Ifnar1", "Plin2", "Vcan", "Olr1", "Fabp5", "Ccl2", "Clec4a2"], ncols=3, frameon=False, wspace=0.1)

In [ ]:
with rc_context({"figure.figsize": (4, 3)}):
 plt.rcParams['font.family'] = 'Arial'
 plt.rcParams['font.size'] = 12
 sc.pl.umap(adata, color=["Cell Types"], ncols=2, frameon=False, wspace=0.1, title="", save="poster_umap_markers_Ms.pdf")

In [ ]:
with rc_context({"figure.figsize": (3, 2.1), "font.size": 12}):
 sc.pl.dotplot(adata[adata.obs["Cell Types"] == "TLF-Ms"], ["Trem2", "Stat1", "Stat2", "Ifnar1"], "Treatment", dendrogram=False, figsize=(3, 2.1),cmap='viridis', save="Fig3_dotplot_MsTLF.pdf")

In [ ]:
with rc_context({"figure.figsize": (3, 2.1), "font.size": 12}):
 sc.pl.dotplot(adata[adata.obs["Cell Types"] == "Trem2-Gpnmb-Ms"], ["Trem2","Stat1", "Stat2", "Ifnar1"], "Treatment", dendrogram=False, figsize=(3, 2.1),cmap='viridis', save="Fig3_dotplot_MsTrem2GPnmb.pdf")

In [ ]:
with rc_context({"figure.figsize": (4, 2.2), "font.size": 12}):
    sns.set_style("white")
    #plt.rcParams['font.family'] = 'Arial'
    #plt.rcParams['font.size'] = 12
    tmp = pd.crosstab(adata.obs["Cell Types"],adata.obs['Treatment'], normalize='columns').T.plot(kind='bar', stacked=True, legend=None)
        #plt.ylabel('Treatment')
    plt.xlabel(None)
    plt.xticks(rotation=0)
    plt.title('Cell composition')
    plt.savefig("figures/Cell_composition_Ms.pdf") # save as png

In [ ]:
with rc_context({"figure.figsize": (3.8, 2), "font.size": 12}):
    sns.set_style("white")
    sc.pl.violin(adata[adata.obs["Cell Types"] == "TLF-Ms"], ["Trem2","Stat1", "Stat2", "Ifnar1"], groupby="Treatment", xlabel=None, multi_panel=True, palette=color_key, order=["HFD+vehicle", "HFD+msR4M-L1"], save="Fig3_violin_markers_Ms_tlf.pdf")
    sc.pl.violin(adata[adata.obs["Cell Types"] == "Trem2-Gpnmb-Ms"], ["Trem2","Stat1", "Stat2", "Ifnar1"], groupby="Treatment", xlabel=None, multi_panel=True, palette=color_key, order=["HFD+vehicle", "HFD+msR4M-L1"], save="Fig3_violin_markers_2_ms_trem2_gpnmb.pdf")


In [ ]:
adata.obs["Treatment"]

In [ ]:
adatatemp=adata.copy()

In [ ]:
adatatemp

In [ ]:
marker_genes_Wirka2019 = {
"Zernecke2022-Ms subset": ["Retnla", "Cd226","Ccr2","Chil3" ,"Ly6c2", "Ace", "Treml4","Isg15", "Ifit1", "Irf7", "Oasl2", "Apoc1","Psap",  "Ctsd", "Lipa","Lpl", "Fth1", "Ftl1","Ctss", "Apoe","Fabp5", "Atp6v0d2", "Gpnmb", "Slamf9", "Cd72", "Ch25h", "Hexb", "Capg","Cd9","Itgax","Spp1","Trem2","Nlrp3", "Ifrd1", "Il1b", "Kdm6b","Tmem176b","Tmem176a","H2-Eb1", "H2-Aa", "Jun", "Fos", "Atf3","Stab1", "Pf4", "Cd163", "Maf","Cd209f", "Cd209d", "Cd209g", "Fxyd2", "Mrc1","Timd4", "Lyve1"],
"TLF-Ms":["Timd4","Lyve1","Folr2","Pf4","Cbr2","Maf"],
"Trem2-Gpnmb-Ms":["Acp5", "Gngt2","Mpeg1","Trem2","Cd9","Fabp5", "Gpnmb", "Ctsd"],

  

}

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (3,3)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3, vmin=0, vmax=0.9, 
    size=100, sort_order=False, save="score_Ms.pdf"
    )

In [ ]:
#fig = plt.figure(figsize=(9, 11))
tmp = pd.crosstab(adata.obs['cell_type'],adata.obs['batch'], normalize='columns').T.plot(kind='barh', stacked=True)
tmp.legend(title='Cell Types', bbox_to_anchor=(1.26, 1.02),loc='upper right')
plt.savefig("Ms_subset/deconvolution_Ms.svg") # save as png

In [ ]:
%%R -o find_de_MAST_RE
options(warn=-1)
find_de_MAST_RE <- function(adata_, cell_type) {
    # create a MAST object
    sca <- SceToSingleCellAssay(adata_, class = "SingleCellAssay")
    print("Dimensions before subsetting:")
    print(dim(sca))
    print("")
    # keep genes that are expressed in more than 10% of all cells
    sca <- sca[freq(sca)>0.1,]
    print("Dimensions after subsetting:")
    print(dim(sca))
    print("")
    # add a column to the data which contains scaled number of genes that are expressed in each cell
    cdr2 <- colSums(assay(sca)>0)
    colData(sca)$ngeneson <- scale(cdr2)
    # store the columns that we are interested in as factors
       batch <- factor(colData(sca)$batch)
    # set the reference level
    batch <- relevel(batch,"HFD")
    colData(sca)$batch <- batch
    celltype <- factor(colData(sca)$cell_type)
    colData(sca)$celltype <- celltype
      print("")
    # same for donors (which we need to model random effects)
    #replicate <- factor(colData(sca)$replicate)
    #colData(sca)$replicate <- replicate
    print("")
    # create a group per condition-celltype combination
    colData(sca)$group <- paste0(colData(adata_)$batch, ".", colData(adata_)$cell_type)
    #colData(sca)$group <- factor(colData(sca)$group)
     print("4")
    # define and fit the model
    zlmCond <- zlm(formula = ~ngeneson+group, 
                    sca=sca,
                   
                  
                   ) # to speed up calculations 
                  # to speed up calculations
    print("5")
    
    # perform likelihood-ratio test for the condition that we are interested in    
   summaryCond <- summary(zlmCond, doLRT=paste0('groupHFD_L1.', cell_type))
    # get the table with log-fold changes and p-values
    summaryDt <- summaryCond$datatable
    result <- merge(summaryDt[contrast == paste0('groupHFD_L1.', cell_type) & component == 'H', .(primerid, `Pr(>Chisq)`)], # p-values
                 summaryDt[contrast == paste0('groupHFD_L1.', cell_type) & component == 'logFC', .(primerid, coef)],
                 by = 'primerid') # logFC coefficients
    # MAST uses natural logarithm so we convert the coefficients to log2 base to be comparable to edgeR
    result[,coef:=result[,coef]/log(2)]
    # do multiple testing correction
    result[,FDR:=p.adjust(`Pr(>Chisq)`, 'fdr')]
    result = result[result$FDR<0.01,, drop=F]

    result <- stats::na.omit(as.data.frame(result))
    return(result)
}

In [ ]:
cell_types = list(adata.obs['cell_type'].unique())

# List of conditions
conditions = list(adata.obs["batch"].unique())

# Filter cell_types to include only those found under both conditions
filtered_cell_types = []

for cell_type in cell_types:
    # Check if the cell_type is found in both conditions
    if all(adata.obs[(adata.obs["batch"] == condition) & (adata.obs['cell_type'] == cell_type)].shape[0] > 0 for condition in conditions):
        filtered_cell_types.append(cell_type)

# filtered_cell_types now contains only cell_types found in both conditions
print(filtered_cell_types)
cell_types = filtered_cell_types


In [ ]:

results = {}
for cell_type in cell_types:
    start_time = time.time()  # Record the start time
    adata_cell_type = adata[adata.obs["cell_type"] == cell_type].copy()
    adata_cell_type = prep_anndata(adata_cell_type)  # Adjust this line if needed based on the loop
    res = find_de_MAST_RE(adata_cell_type, cell_type)
    results[cell_type] = res
    res["gene_symbol"] = res["primerid"]
    res["cell_type"] = cell_type
    end_time = time.time()  # Record the end time
    elapsed_time = end_time - start_time

    if len(res) > 0:
        print(f"Calculation for {cell_type} completed in {elapsed_time:.2f} seconds.")
    else:
        print(f"Calculation for {cell_type} failed or produced empty results.")

# Print the total time for all calculations
total_elapsed_time = time.time() - start_time
print(f"Total time for all calculations: {total_elapsed_time:.2f} seconds.")


In [ ]:
type(results)

In [ ]:
Folder = "Ms_subset"
for cell_type, res in results.items():
    start_time = time.time()  # Record the start time
    df = pd.DataFrame(res)
    df = df.sort_values(by='coef', ascending=False)
    df.rename(columns={'coef': 'logFC'}, inplace=True)
    df.rename(columns={'FDR': 'adj_pVal'}, inplace=True)

    # Filter for values greater than 1 or smaller than -1 in the 'logFC' column
    # and values smaller than 0.1 in the 'adj_pVal' column
    filtered_df = df[(df['logFC'] > 1) | (df['logFC'] < -1) & (df['adj_pVal'] < 0.1)]
    filtered_df = filtered_df.sort_values(by='logFC', ascending=False)

    df = df[~(df['primerid'].str.startswith('mt-'))]
    df = df[~(df['primerid'].str.startswith('Gm'))]
    filtered_df = filtered_df[~(filtered_df['primerid'].str.startswith('mt-'))] #filtered_df['primerid'].str.startswith('Gm'))]
    filtered_df = filtered_df[~(filtered_df['primerid'].str.startswith('Gm'))]

    # Replace 'sampleID' with 'cell_type' in the filename
    df.to_excel(f'{Folder}/{cell_type}_DEG.xlsx')
    filtered_df.to_excel(f'{Folder}/filtered_{cell_type}_DEG.xlsx')
    df.to_csv(f'{Folder}/{cell_type}_DEG.csv')
    filtered_df.to_csv(f'{Folder}/filtered_{cell_type}_DEG.csv')

    end_time = time.time()  # Record the end time
    elapsed_time = end_time - start_time

    if len(res) > 0:
        print(f"Processed and saved results for {cell_type} in {elapsed_time:.2f} seconds.")
    else:
        print(f"Processing for {cell_type} failed or produced empty results.")

# Print the total time for all calculations
total_elapsed_time = time.time() - start_time
print(f"Total time for all calculations: {total_elapsed_time:.2f} seconds.")


In [ ]:
cell_types


In [ ]:
import os
import pandas as pd
import rpy2.robjects as robjects
from IPython.display import display, HTML


# Set the directory where your CSV files are located
csv_folder = 'Ms_subset'

# Define the custom file extension (e.g., '.custom_extension')

# List all CSV files in the directory that don't start with "filtered"
csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv') and not f.startswith('filtered')]

# Import necessary R packages
robjects.r("library(EnhancedVolcano)")
robjects.r("library(ggplot2)")

# Loop through the CSV files and generate EnhancedVolcano plots
for csv_file in csv_files:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(os.path.join(csv_folder, csv_file))

    # Convert the "logFC" column to a numeric type
    df['logFC'] = pd.to_numeric(df['logFC'])
    
    # Split csv_file at "_"
    filename_parts = csv_file.split("_")
    
    # Create the title from relevant parts
    title = " ".join(filename_parts[:-1])  # Join parts except the last (assumed to be the extension)

    # Send the DataFrame to R
    robjects.globalenv['df'] = robjects.pandas2ri.py2rpy(df)

    # Run the EnhancedVolcano code in R
    robjects.r(f"""
    options(warn=-1)
    p1 <- EnhancedVolcano(df,
      lab = df$primerid,
      x = 'logFC',
      y = 'adj_pVal',
      title = 'DEG in {title}',
      pCutoff = 10e-5,
      FCcutoff = 1,
      pointSize = 3.0,
      labSize = 6.0,
      drawConnectors = TRUE,
      widthConnectors = 0.75,
      subtitle = "Left: Down in L1 mice, Right: Up in L1 mice ")

    # Save the file with the custom extension
    ggsave(sprintf("{Folder}/{title}.pdf"), plot = p1)
     ggsave(sprintf("{Folder}/{title}.svg"), plot = p1)
    """)
     # Append the SVG path to the list
svg_paths = []  # Initialize an empty list before the loop
    
svg_paths.append(f"DEG_Volcano_plot_new/{title}.svg")

print("Volcano plots generated and saved for all CSV files with the custom extension.")



   

In [ ]:
import os
from IPython.display import display, HTML, SVG


# Set the directory where your SVG files are located
svg_folder = Folder

# List all SVG files in the directory
svg_files = [f for f in os.listdir(svg_folder) if f.endswith('.svg')]

# Display the SVG plots side by side using the SVG class
for svg_file in svg_files:
    svg_path = os.path.join(svg_folder, svg_file)
    svg_image = SVG(filename=svg_path)
    display(svg_image)



In [ ]:
import pandas as pd
import os

# Define the directory where your CSV files are located
folder_path = r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT\Ms_subset'

# Initialize an empty dictionary to store the DataFrames
dataframes = {}

# Loop through the files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):  # Check if the file is a CSV file
        file_path = os.path.join(folder_path, filename)  # Get the full file path
        key = os.path.splitext(filename)[0]  # Use the filename (without extension) as the key
        df = pd.read_csv(file_path)  # Read the CSV file into a DataFrame
        dataframes[key] = df  # Add the DataFrame to the dictionary

        

dataframes = {key: df for key, df in dataframes.items() if not key.startswith('filtered_')}

# Now, 'dataframes' is a dictionary where keys are filenames, and values are DataFrames


In [ ]:
%%R -o Go_function
library(openxlsx)
options(warn=-1)
library(clusterProfiler)
library(org.Mm.eg.db)
library(AnnotationDbi)
library(ggplot2)

Go_function <- function(df, ctname, filename1, filename2, filename3, filename4) {
  df_down <- rownames(df[df$logFC < -0.5,])
  GO_results_down <- enrichGO(gene = df_down, OrgDb = "org.Mm.eg.db", keyType = "SYMBOL", ont = "BP")
  as.data.frame(GO_results_down)
  
  # Use dotplot for lollipop plot
  dotplot(GO_results_down, showCategory = 15, title = paste("GO BP terms down in L1 mice", ctname), font.size = 10)
  ggsave(filename1, width = 6, height = 8)
  
  GO_results_down <- GO_results_down[order(GO_results_down$'p.adjust', decreasing = F),]
  write.xlsx(GO_results_down, filename2)
  
  df_up <- rownames(df[df$logFC > 0.5,])
  GO_results_up <- enrichGO(gene = df_up, OrgDb = "org.Mm.eg.db", keyType = "SYMBOL", ont = "BP")
  as.data.frame(GO_results_up)
  
  # Use dotplot for lollipop plot
  dotplot(GO_results_up, showCategory = 15, title = paste("GO BP terms up in L1 mice", ctname), font.size = 10)
  ggsave(filename3, width = 6, height = 8)
  
  GO_results_up <- GO_results_up[order(GO_results_up$'p.adjust', decreasing = F),]
  write.xlsx(GO_results_up, filename4)
}


In [ ]:
for key in dataframes:
    print(key)

In [ ]:
from IPython.display import display, SVG, HTML
import matplotlib.pyplot as plt

# Enable inline plotting
%matplotlib inline


# Access the first entry in the dataframes dictionary

#for key in dataframes:

    #print(key)
varct= 'TLF-Ms_DEG'
lablect = varct.split('_')
ctname=lablect[0]
filename1 = f"{Folder}/GO_{ctname}_down.svg"
filename2 = f"{Folder}/GO_{ctname}_down.xlsx"
filename3 = f"{Folder}/GO_{ctname}_up.svg"
filename4 = f"{Folder}/GO_{ctname}_up.xlsx"
df=dataframes[varct]
df = df.set_index('gene_symbol')
Go_function(df,ctname,filename1,filename2,filename3,filename4)
    # Load and display SVG images
svg1 = SVG(filename1)
svg3 = SVG(filename3)
    
    # Display the SVG images side by side
display(HTML(f'<table><tr><td width="600" height="300">{svg1.data}</td><td width="600" height="300">{svg3.data}</td></tr></table>'))

varct= 'Trem2-Gpnmb-Ms_DEG'
lablect = varct.split('_')
ctname=lablect[0]
filename1 = f"{Folder}/GO_{ctname}_down.svg"
filename2 = f"{Folder}/GO_{ctname}_down.xlsx"
filename3 = f"{Folder}/GO_{ctname}_up.svg"
filename4 = f"{Folder}/GO_{ctname}_up.xlsx"
df=dataframes[varct]
df = df.set_index('gene_symbol')
Go_function(df,ctname,filename1,filename2,filename3,filename4)
    # Load and display SVG images
svg1 = SVG(filename1)
svg3 = SVG(filename3)
    
    # Display the SVG images side by side
display(HTML(f'<table><tr><td width="600" height="300">{svg1.data}</td><td width="600" height="300">{svg3.data}</td></tr></table>'))

    
   

In [ ]:
import pandas as pd
import os

# Define the directory where your CSV files are located
folder_path = r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT\DEG_List_filt'

# Initialize an empty dictionary to store the DataFrames
dataframes = {}

# Loop through the files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):  # Check if the file is a CSV file
        file_path = os.path.join(folder_path, filename)  # Get the full file path
        key = os.path.splitext(filename)[0]  # Use the filename (without extension) as the key
        df = pd.read_csv(file_path)  # Read the CSV file into a DataFrame
        dataframes[key] = df  # Add the DataFrame to the dictionary

        

dataframes = {key: df for key, df in dataframes.items() if not key.startswith('filtered_')}

# Now, 'dataframes' is a dictionary where keys are filenames, and values are DataFrames


In [ ]:
for key in dataframes:
    print(key)

In [ ]:

df=dataframes['Ms_DEG_DEG']


In [ ]:
    
%%R -i df 
    library(EnhancedVolcano)
    library(ggplot2)

    p1<-EnhancedVolcano(df,
     lab = df$primerid,       
         x = 'logFC',
        y = 'adj_pVal',
        title = 'DEG in the Macrophage cluster',
        pCutoff = 0.001,
        selectLab = c('Stat1','Stat2', 'Cd74', 'Ifi203', 'Nlrc5', 'Ifnar1', 'Plcg2', 'Sp100', 'Trim30a','Spp1','Fth1','Mgp','Mfge8','Igf2r','Tmtm165','Lgals1','Lgals3','Gsn','Prkg1','Sgk1', 'Mmp12', 'Fabp5', 'Gpnmb','Ftl1', 'Ly86', 'CD63' ), #Lable the genes of interest
        FCcutoff = 1,
        pointSize = 2.0,
        labSize = 4.0,
        drawConnectors = TRUE,
        widthConnectors = 0.75)
show(p1)


ggsave("Volcan_plot_filt/Ms_DEG.pdf")
ggsave("Volcan_plot_filt/Ms_DEG.svg")

